# Music by Prompt: первый запуск

1. Выберите **Среда выполнения → Сменить среду выполнения → GPU**.
2. Загрузите на Google Drive готовые данные из локального проекта:
   - `data/model/features.npz` → `MyDrive/music_prompt/data/model/features.npz`
   - папку `data/model/pilot/` целиком → `MyDrive/music_prompt/data/model/pilot/`
3. Запустите ячейки по порядку. Код загрузится из GitHub; исходные диалоги, Onion и recsys не нужны.

По умолчанию запускаются **10 проверочных optimizer steps**. Для основного пилота задайте `RUN_NAME="encdec"`, `STEPS=1284`, `EVAL_EVERY=100`, `VAL_TURNS=5000` — это две эпохи на текущем пилоте.


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/music_prompt')
RUN_NAME = 'sanity'
STEPS = 10
EVAL_EVERY = 10
VAL_TURNS = 500
ITEM_FEATURES = 'audio,text,tags'
RESUME = False  # True: продолжить last.pt того же запуска с прежними настройками


In [ ]:
import subprocess
import sys

ROOT = Path('/content/music_prompt')
if not ROOT.exists():
    subprocess.run([
        'git', 'clone', '--single-branch', '--branch', 'research/encoder-decoder-embeddings',
        'https://github.com/taisia-zlotnikova/music_recsys_dataset.git', str(ROOT)
    ], check=True)
%cd /content/music_prompt
%pip install -r twotower/requirements.txt


Если установка попросит перезапустить сессию Python, перезапустите её и снова выполните ячейки. Первый запуск скачает предобученную Qwen из Hugging Face.


In [ ]:
import shutil
import torch

assert torch.cuda.is_available(), 'Включите GPU в настройках среды Colab'
print('GPU:', torch.cuda.get_device_name(0))
source = DRIVE_ROOT / 'data/model'
local = ROOT / 'data/model'
assert (source / 'features.npz').is_file(), 'Загрузите features.npz на Drive'
assert (source / 'pilot/manifest.json').is_file(), 'Загрузите папку pilot целиком'
local.mkdir(parents=True, exist_ok=True)
shutil.copy2(source / 'features.npz', local / 'features.npz')
shutil.copytree(source / 'pilot', local / 'pilot', dirs_exist_ok=True)
print('Данные скопированы на локальный диск Colab')


In [ ]:
out = DRIVE_ROOT / 'runs' / RUN_NAME
command = [
    sys.executable, '-m', 'twotower.train', '--device', 'cuda',
    '--steps', str(STEPS), '--eval-every', str(EVAL_EVERY),
    '--val-turns', str(VAL_TURNS), '--item-features', ITEM_FEATURES,
    '--out', str(out)
]
if RESUME:
    command += ['--resume', str(out / 'last.pt')]
subprocess.run(command, cwd=ROOT, check=True)
print('Результаты на Drive:', out)


В папке запуска на Drive сохраняются `config.json`, `metrics.jsonl`, `best.pt`, `last.pt`. Checkpoint записывается при validation и на последнем шаге.

Для продолжения после обрыва оставьте прежние `RUN_NAME`, `ITEM_FEATURES`, `VAL_TURNS`, установите `RESUME=True`, а `STEPS` — нужное **суммарное** число шагов. Для нового эксперимента задайте новый `RUN_NAME` и `RESUME=False`.

10 шагов проверяют работоспособность, а не качество. Инструкции для оценки, просмотра примеров и дополнительных признаков: [twotower/README.md](https://github.com/taisia-zlotnikova/music_recsys_dataset/blob/research/encoder-decoder-embeddings/twotower/README.md).
